In [1]:
import kaggle as kg
import os
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import seaborn as sns
import cv2
from keras.applications import VGG16
from matplotlib.pyplot import figure
from glob import glob
from keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator,load_img,img_to_array
from keras.layers import Input, Dense,Conv2D,MaxPooling2D,Flatten,Dropout,BatchNormalization,GlobalAveragePooling2D
from keras.models import Model
from keras.optimizers import RMSprop,SGD
plt.style.use("ggplot")

2024-09-16 20:26:43.836171: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-09-16 20:26:43.908777: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-16 20:26:44.413858: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-16 20:26:44.418847: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-09-16 20:26:45.865371: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

In [2]:
def count_folders(directory):
    folder_count = 0
    for entry in os.scandir(directory):
        if entry.is_dir():
            folder_count += 1
    return folder_count

def count_images(directory):
    images_count = 0
    
    for root, dir, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(("jpeg","png","jpg")):
                images_count = images_count + 1 
    return images_count
# Function to count the total number of files in a directory
def total_files(folder_path):
    num_files = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
    return num_files


train = "/home/thasin/plant_disease/dataset/train/train/train"
valid = "/home/thasin/plant_disease/dataset/valid"
test = "/home/thasin/plant_disease/dataset/test/test"

# Count the number of folders in train and valid directories
total_folders_train = count_folders(train)
total_folders_valid = count_folders(valid)

# Count the total number of files in the test directory
total_files_test = total_files(test)

print(f"Total number of folders in Train: {total_folders_train} and images in the folder is {count_images(train)}")
print(f"Total number of folders in Valid: {total_folders_valid} and images in that folder is {count_images(valid)}")
print(f"Total number of files in Test: {total_files_test} and images in that folder is {count_images(test)}")


Total number of folders in Train: 38 and images in the folder is 70295
Total number of folders in Valid: 38 and images in that folder is 17572
Total number of files in Test: 33 and images in that folder is 33


In [3]:
# # Load the train dataset
# train_dataset = tf.keras.utils.image_dataset_from_directory(
#     train,
#     labels="inferred",
#     label_mode="categorical", 
#     color_mode="rgb",
#     batch_size=32,
#     image_size=(256, 256),  # resize images to 256x256
#     shuffle=True,
#     seed=None,
# )


Found 70295 files belonging to 38 classes.


In [5]:
# # Load the validation dataset
# valid_dataset = tf.keras.utils.image_dataset_from_directory(
#     valid,
#     labels="inferred",
#     label_mode="categorical",
#     color_mode="rgb",
#     batch_size=32,
#     image_size=(256, 256),
#     shuffle=True,
#     seed=None,
# )


Found 17572 files belonging to 38 classes.


In [ ]:
# for images, labels in train_dataset.take(2):
#     print(images,images.shape)
#     print(labels,labels.shape)

In [11]:
# def train_test_df(img_full_path):
#     img_complete_path = list()
#     label_complete_path = list()
    
#     for single_image_path in pathlib.Path(img_full_path).glob("*"):
#            img_path = str(single_image_path)
           
#            img_labell_path = os.join(labels_full_path,str(single_image_path)).split("/")[-1].split(".")[0]
           
#            img_complete_path.append(img_path)
           
#            label_complete_path.append(img_labell_path)
        
#     return pd.DataFrame(data={"Img_path":img_full_path,"Label":labels_full_path}) 

In [3]:
def train_test_df(path,is_test=False):

    img_path = list()
    img_label = list()

    if is_test:
        # Iterate through each image in the test folder
        for img_file_path in pathlib.Path(path).glob("*.JPG"):
            img_path.append(str(img_file_path))  # Store the image path
            
            # Assuming label is derived from the filename, for example, before the first underscore
            img_label.append(str(img_file_path.stem).split("_")[0])
            
    else:        
       
       for single_class_dir_path in pathlib.Path(path).glob("*"):
   

       
           if single_class_dir_path.is_dir():
               
               label = single_class_dir_path.stem
               
               for img_file_path in pathlib.Path(single_class_dir_path).glob("*.[jJ][pP][gG]"):
       
                    img_path.append(str(img_file_path))  # Store the image path


       
                    # Assuming label is derived from the filename, for example, before the first underscore
       
                    img_label.append(label)




       
               # img_path.append(str(single_class_img_path))
       
               # #print(str(single_class_img_path).split("/")[-2].split("_")[-1])
       
               # img_label.append(str(single_class_img_path).split("/")[-2].split("_")[-1])
   

       
    return pd.DataFrame(data={"img_path":img_path,"label":img_label})

In [4]:
train_path = "/home/thasin/plant_disease/dataset/train/train/train"
valid_path = "/home/thasin/plant_disease/dataset/valid"
test_path = "/home/thasin/plant_disease/dataset/test/test"

In [5]:
train_df = train_test_df(train_path,is_test=False)
valid_df = train_test_df(valid_path,is_test=False)
test_df = train_test_df(test_path,is_test=True)

In [6]:
train_df.shape

(70295, 2)

In [7]:
train_df.head(10)

,img_path,label
0,/home/thasin/plant_disease/dataset/train/train...,Corn_(maize)___Common_rust_
1,/home/thasin/plant_disease/dataset/train/train...,Corn_(maize)___Common_rust_
2,/home/thasin/plant_disease/dataset/train/train...,Corn_(maize)___Common_rust_
3,/home/thasin/plant_disease/dataset/train/train...,Corn_(maize)___Common_rust_
4,/home/thasin/plant_disease/dataset/train/train...,Corn_(maize)___Common_rust_
5,/home/thasin/plant_disease/dataset/train/train...,Corn_(maize)___Common_rust_
6,/home/thasin/plant_disease/dataset/train/train...,Corn_(maize)___Common_rust_
7,/home/thasin/plant_disease/dataset/train/train...,Corn_(maize)___Common_rust_
8,/home/thasin/plant_disease/dataset/train/train...,Corn_(maize)___Common_rust_
9,/home/thasin/plant_disease/dataset/train/train...,Corn_(maize)___Common_rust_


In [8]:
test_df.shape

(33, 2)

In [9]:
valid_df.shape

(17572, 2)

In [10]:
valid_df.head(10)

,img_path,label
0,/home/thasin/plant_disease/dataset/valid/Corn_...,Corn_(maize)___Common_rust_
1,/home/thasin/plant_disease/dataset/valid/Corn_...,Corn_(maize)___Common_rust_
2,/home/thasin/plant_disease/dataset/valid/Corn_...,Corn_(maize)___Common_rust_
3,/home/thasin/plant_disease/dataset/valid/Corn_...,Corn_(maize)___Common_rust_
4,/home/thasin/plant_disease/dataset/valid/Corn_...,Corn_(maize)___Common_rust_
5,/home/thasin/plant_disease/dataset/valid/Corn_...,Corn_(maize)___Common_rust_
6,/home/thasin/plant_disease/dataset/valid/Corn_...,Corn_(maize)___Common_rust_
7,/home/thasin/plant_disease/dataset/valid/Corn_...,Corn_(maize)___Common_rust_
8,/home/thasin/plant_disease/dataset/valid/Corn_...,Corn_(maize)___Common_rust_
9,/home/thasin/plant_disease/dataset/valid/Corn_...,Corn_(maize)___Common_rust_


In [11]:
convert_to_int = dict(zip(train_df["label"].unique(),range(len(train_df["label"].unique()))))

In [12]:
convert_to_int

{'Corn_(maize)___Common_rust_': 0,
 'Potato___Late_blight': 1,
 'Pepper,_bell___healthy': 2,
 'Tomato___Leaf_Mold': 3,
 'Blueberry___healthy': 4,
 'Cherry_(including_sour)___healthy': 5,
 'Squash___Powdery_mildew': 6,
 'Orange___Haunglongbing_(Citrus_greening)': 7,
 'Raspberry___healthy': 8,
 'Apple___Black_rot': 9,
 'Potato___Early_blight': 10,
 'Tomato___Septoria_leaf_spot': 11,
 'Potato___healthy': 12,
 'Cherry_(including_sour)___Powdery_mildew': 13,
 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 14,
 'Peach___Bacterial_spot': 15,
 'Strawberry___healthy': 16,
 'Apple___Apple_scab': 17,
 'Peach___healthy': 18,
 'Pepper,_bell___Bacterial_spot': 19,
 'Grape___Esca_(Black_Measles)': 20,
 'Tomato___Late_blight': 21,
 'Grape___healthy': 22,
 'Soybean___healthy': 23,
 'Apple___healthy': 24,
 'Corn_(maize)___Northern_Leaf_Blight': 25,
 'Corn_(maize)___healthy': 26,
 'Tomato___Spider_mites Two-spotted_spider_mite': 27,
 'Tomato___Bacterial_spot': 28,
 'Tomato___healthy': 29,
 'Tomato___Early

In [13]:
range(len(train_df['label'].unique()))

range(0, 38)

In [14]:
train_df["label"].replace(to_replace=convert_to_int.keys(),value=convert_to_int.values(),inplace=True)

In [15]:
train_df.tail(2000)

,img_path,label
68295,/home/thasin/plant_disease/dataset/train/train...,36
68296,/home/thasin/plant_disease/dataset/train/train...,36
68297,/home/thasin/plant_disease/dataset/train/train...,36
68298,/home/thasin/plant_disease/dataset/train/train...,36
68299,/home/thasin/plant_disease/dataset/train/train...,36
...,...,...
70290,/home/thasin/plant_disease/dataset/train/train...,37
70291,/home/thasin/plant_disease/dataset/train/train...,37
70292,/home/thasin/plant_disease/dataset/train/train...,37
70293,/home/thasin/plant_disease/dataset/train/train...,37


In [16]:
valid_df.replace(to_replace=convert_to_int.keys(),value=convert_to_int.values(),
                     inplace=True)

In [17]:
valid_df.tail(2000)

,img_path,label
15572,/home/thasin/plant_disease/dataset/valid/Corn_...,33
15573,/home/thasin/plant_disease/dataset/valid/Corn_...,33
15574,/home/thasin/plant_disease/dataset/valid/Corn_...,33
15575,/home/thasin/plant_disease/dataset/valid/Corn_...,33
15576,/home/thasin/plant_disease/dataset/valid/Corn_...,33
...,...,...
17567,/home/thasin/plant_disease/dataset/valid/Straw...,37
17568,/home/thasin/plant_disease/dataset/valid/Straw...,37
17569,/home/thasin/plant_disease/dataset/valid/Straw...,37
17570,/home/thasin/plant_disease/dataset/valid/Straw...,37


In [18]:
Y_true_train = to_categorical(y=train_df["label"],num_classes=38)
Y_true_test = to_categorical(y=valid_df["label"],num_classes=38)

In [19]:
Y_true_train

array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.]], dtype=float32)

In [20]:
Y_true_train.shape, Y_true_test.shape

((70295, 38), (17572, 38))

In [21]:
def plant_disease_cnn():

    vgg16 = VGG16(include_top=False,input_shape=(256,256,3),weights="imagenet",pooling=None)
    vgg16.trainable = False
    input_to_vgg16 = vgg16.input
    vgg16_output = vgg16.output
    
    x = Conv2D(filters=64, kernel_size=(3, 3), activation='relu')(vgg16_output)
    x = BatchNormalization()(x)
    
    x = GlobalAveragePooling2D()(x)
    
    x = Dense(128,activation='relu')(x)
    x = Dense(64,activation='relu')(x)
    
    vgg16_output = Dense(38,activation='softmax')(x)

    return Model(inputs=[input_to_vgg16],outputs=[vgg16_output])

In [22]:
model = plant_disease_cnn()
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 256, 256, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 256, 256, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 256, 256, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 128, 128, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 128, 128, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 128, 128, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 64, 64, 128)       0     

In [23]:
def custom_data_generator(data_df,Y_true,mb_size):
    
    for time_step in range(data_df.shape[0]//mb_size):
        X_mb = list()
        for img_path in data_df.iloc[time_step*mb_size:(time_step+1)*mb_size,0]:
            img_np_array = plt.imread(img_path)
            reshaped_np_array = cv2.resize(img_np_array, (256, 256))
            X_mb.append(reshaped_np_array)
        X_mb = np.array(X_mb)
        Y_true_mb = Y_true[time_step*mb_size:(time_step+1)*mb_size]
        
        yield X_mb, Y_true_mb

In [29]:
epochs = 50
training_data_mb_size = 32
testing_data_mb_size = 32

In [30]:
def loss_fn(Y_true_mb,Y_pred_mb):
    return tf.reduce_mean(tf.keras.losses.categorical_crossentropy(y_true=Y_true_mb,y_pred=Y_pred_mb))
optimizer = SGD()

In [31]:
@tf.function
def training_step(X_train_mb,Y_true_train_mb):

    with tf.GradientTape() as tape:
            
        Y_pred_train_mb = model(X_train_mb, training=True)
        training_loss = loss_fn(Y_true_train_mb, Y_pred_train_mb)

    grads = tape.gradient(training_loss, model.trainable_weights)
    optimizer.apply_gradients(zip(grads, model.trainable_weights))

    train_acc_metric.update_state(Y_true_train_mb,Y_pred_train_mb)

    return training_loss

In [32]:
@tf.function
def testing_forward_pass(X_test_mb,Y_true_test_mb):

    Y_pred_test_mb = model(X_test_mb,training=False)
    testing_loss = loss_fn(Y_true_test_mb,Y_pred_test_mb)
    test_acc_metric.update_state(Y_true_test_mb,Y_pred_test_mb)

    return testing_loss

In [33]:
train_acc_metric = tf.keras.metrics.CategoricalAccuracy()
test_acc_metric = tf.keras.metrics.CategoricalAccuracy()

for epoch in range(epochs):

    training_data_generator = custom_data_generator(train_df,Y_true_train,32)

    for time_step, (X_train_mb, Y_true_train_mb) in enumerate(training_data_generator):
        training_loss = training_step(X_train_mb,Y_true_train_mb)

        if (time_step+1) % 50 == 0:
            print("Epoch %d, Time Step %d, Training loss for one mini batch: %.4f"
            % (epoch+1, time_step+1, float(training_loss)))
            
    training_acc = train_acc_metric.result()    
    print("Epoch %d, Training Accuracy: %.2f" % (epoch+1,float(training_acc)))
    train_acc_metric.reset_states()

    testing_data_generator = custom_data_generator(valid_df,Y_true_test,testing_data_mb_size)

    for X_test_mb, Y_true_test_mb in testing_data_generator:
        testing_loss = testing_forward_pass(X_test_mb,Y_true_test_mb)

    print("\nEpoch %d, Testing Loss for last mini batch: %.4f" % (epoch+1,float(testing_loss)))
    testing_acc = test_acc_metric.result()
    print("Epoch %d, Testing Accuracy: %.2f" % (epoch+1,float(testing_acc)))
    test_acc_metric.reset_states()

    print("\n\n")


Epoch 1, Time Step 50, Training loss for one mini batch: 0.3098
Epoch 1, Time Step 100, Training loss for one mini batch: 0.5789


KeyboardInterrupt: 